# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Iqra411/lyrank-ML-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*
 Traffic-like fields are heavily right-skewed, as expected for web metrics
(a few giant pages, a long tail of tiny ones):

| Field | mean | median | p90 | p99 | max | skew |
|---|---|---|---|---|---|---|
| impressions_90d | 5,200 | 731 | 12,136 | 73,506 | 517,715 | 11.4 |
| clicks_90d | 16.1 | 1.0 | 32.0 | 253.0 | 4,178 | 18.3 |
| ctr | 0.5 | 0.1 | 0.7 | 8.3 | 100.0 | 17.4 |
| avg_position | 16.3 | 10.8 | 36.8 | 69.9 | 245.0 | 2.0 |
| word_count | 2,310 | 2,605 | 4,719 | 7,090 | 9,546 | 0.5 |

impressions_90d, clicks_90d, and ctr are extremely heavy-tailed (skew 11-18) --
mean is 5-16x the median in the traffic columns. Per auditing-signals/SKILL.md,
plain Pearson correlation on these raw values would be dominated by a handful
of giant pages, so every traffic-involving test below uses log1p or rank
(Spearman) correlation instead of raw Pearson. word_count and avg_position
are much closer to symmetric (skew 0.5 and 2.0) and don't need the same
treatment.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('https://raw.githubusercontent.com/Iqra411/lyrank-ML-internship/main/data/raw/content_refresh_anonymized.csv')

numeric_fill_zero = [
    "search_volume","competition","cpc","word_count","char_count",
    "impressions_90d","clicks_90d","pageviews_90d","sessions_90d","users_90d",
    "engaged_sessions_90d","ai_sessions_90d","scroll_events_90d",
    "days_with_impressions","days_with_sessions","impressions_last_30d",
    "clicks_last_30d","sessions_last_30d","impressions_prev_30d",
    "clicks_prev_30d","sessions_prev_30d","content_age_days","age_tier_order",
    "days_since_last_update","ctr","avg_position","engagement_rate",
    "scroll_rate","ai_traffic_pct","trend_pct",
]
for c in numeric_fill_zero:
    df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat_cols = ["competition_level","content_type","main_intent","provider_used","model_used",
            "age_tier","freshness_tier","word_count_tier","char_count_tier",
            "impression_tier","position_tier","trend_direction"]
for c in cat_cols:
    df[c] = df[c].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

for col in ["impressions_90d","word_count","ctr","avg_position","clicks_90d"]:
    s = df[col]
    print(f"{col}: mean={s.mean():.1f} median={s.median():.1f} p90={s.quantile(.9):.1f} p99={s.quantile(.99):.1f} max={s.max():.1f} skew={stats.skew(s):.2f}")

impressions_90d: mean=5200.4 median=731.0 p90=12136.4 p99=73505.8 max=517715.0 skew=11.38
word_count: mean=2310.2 median=2605.0 p90=4719.2 p99=7090.1 max=9546.0 skew=0.49
ctr: mean=0.5 median=0.1 p90=0.7 p99=8.3 max=100.0 skew=17.44
avg_position: mean=16.3 median=10.8 p90=36.8 p99=69.9 max=245.0 skew=1.98
clicks_90d: mean=16.1 median=1.0 p90=32.0 p99=253.0 max=4178.0 skew=18.34


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*  
SIGNAL 1: "Longer pages get more traffic"
Test: word_count vs. impressions_90d, n=22,301 (rows with measured word_count).
Raw Pearson 0.163 (weak, distorted by heavy tail) -> log1p(impressions)
Pearson 0.337, Spearman (rank) 0.299 -- both moderate positive. Median
impressions climb monotonically across every word_count_tier: <1000 (4),
1000-2000 (172), 2000-3500 (997), 3500+ (1,340).
VERDICT: CONFIRMED (directional). Longer pages do observably get more
traffic in this dataset -- moderate, not dramatic, and correlation only.

SIGNAL 2: "Better position gets higher CTR"
Test: avg_position vs. ctr, restricted to impressions_90d >= 100 (volume
floor, per data dictionary's own warning about top_3 CTR needing a volume
floor) and avg_position > 0 (excluding "no data" rows). Spearman -0.362 --
confirms the overall direction (better/lower position, higher CTR).
BUT median CTR by tier: top_3 0.19% < page_1 0.23% < striking 0.15%
page_3_5 0.06% < deep 0.00% -- top_3 is NOT the highest, page_1 is,
breaking strict monotonicity right at the top (n=533 for top_3, well
above the 50-row floor, so this isn't small-sample noise).
VERDICT: MIXED. Direction holds in aggregate (rank correlation, and the
bottom three tiers are perfectly monotonic), but the popular belief that
"position 1-3 always beats position 4-10 on CTR" doesn't hold here --
page_1 (positions 4-10) actually out-CTRs top_3 in this slice.

SIGNAL 3: "Fresher content declines less"
Test: decline_rate by freshness_tier. 0-30 days: 51.1% (n=20,480) vs.
91-180 days: 61.1% (n=9,171) -- the two large buckets confirm the
direction. But 31-90 days: 58.9% (n=175) and 181+ days: 47.1% (n=174)
break monotonicity -- both are small (above the 50-row floor, but far
smaller than the two big buckets, so trust them less).
VERDICT: MIXED. Confirmed on the two buckets with real sample size;
the smaller middle and tail buckets don't extend the pattern cleanly,
and I'm not overriding a floor-respecting result just because the story
would be cleaner.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("--- Signal 1: word_count vs impressions_90d ---")
wc = df[df["word_count"] > 0].copy()
wc["log_impr"] = np.log1p(wc["impressions_90d"])
print("n=", len(wc))
print("Pearson raw:", round(wc["word_count"].corr(wc["impressions_90d"]), 3))
print("Pearson log-impr:", round(wc["word_count"].corr(wc["log_impr"]), 3))
print("Spearman:", round(wc["word_count"].corr(wc["impressions_90d"], method="spearman"), 3))
print(wc.groupby("word_count_tier", observed=True)["impressions_90d"].agg(["median","count"]))

print("\n--- Signal 2: avg_position vs ctr (volume floor: impressions_90d>=100) ---")
pos = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 100)].copy()
print(pos.groupby("position_tier", observed=True).agg(median_ctr=("ctr","median"), n=("ctr","size")).reindex(["top_3","page_1","striking","page_3_5","deep"]))
print("Spearman(avg_position, ctr):", round(pos["avg_position"].corr(pos["ctr"], method="spearman"), 3))

print("\n--- Signal 3: freshness_tier vs decline_rate ---")
print(df.groupby("freshness_tier", observed=True).agg(decline_rate=("is_declining_label","mean"), n=("is_declining_label","size")))

--- Signal 1: word_count vs impressions_90d ---
n= 22301
Pearson raw: 0.163
Pearson log-impr: 0.337
Spearman: 0.299
                 median  count
word_count_tier               
1000-2000         172.0   3780
2000-3500         997.0  11263
3500+            1340.0   6285
<1000               4.0    973

--- Signal 2: avg_position vs ctr (volume floor: impressions_90d>=100) ---
               median_ctr     n
position_tier                  
top_3                0.19   533
page_1               0.23  8633
striking             0.15  5903
page_3_5             0.06  6058
deep                 0.00   879
Spearman(avg_position, ctr): -0.362

--- Signal 3: freshness_tier vs decline_rate ---
                decline_rate      n
freshness_tier                     
0-30                0.511377  20480
181+                0.471264    174
31-90               0.588571    175
91-180              0.611057   9171


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*  
The Week-4 rule's core assumption: freshness_tier == "91-180" AND
impressions_90d >= 300 ("stale but visible") predicts elevated decline.
Test: decline rate for flagged vs. not-flagged rows.

Flagged (stale+visible): n=7,212, decline_rate=61.7%
Not flagged: n=22,788, decline_rate=51.8%
Overall base rate: 54.2%

VERDICT: CONFIRMED, but modest. The flagged group's decline rate is
observably higher than both the unflagged group and the overall base
rate -- a real, floor-clearing lift (n=7,212 is far above the 50-row
floor). But it's a 10-point lift over the unflagged group, not a
dramatic separation -- the rule catches a real pattern, it just isn't
a strong one on its own, which is exactly why the Week-5/6 model (which
uses this signal alongside 25 others) outperforms the single-rule
version at precision@K.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
stale_visible = df[(df["freshness_tier"]=="91-180") & (df["impressions_90d"]>=300)]
not_flagged = df[~((df["freshness_tier"]=="91-180") & (df["impressions_90d"]>=300))]
print("Flagged n=", len(stale_visible), "decline_rate=", round(stale_visible["is_declining_label"].mean(), 3))
print("Not flagged n=", len(not_flagged), "decline_rate=", round(not_flagged["is_declining_label"].mean(), 3))
print("Overall base rate:", round(df["is_declining_label"].mean(), 3))

Flagged n= 7212 decline_rate= 0.617
Not flagged n= 22788 decline_rate= 0.518
Overall base rate: 0.542


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.* A content team using position tiers as a proxy for "good enough, don't
touch it" should know that page_1 (positions 4-10) measurably out-CTRs
top_3 (positions 1-3) in this data -- so a top_3 ranking alone isn't a
signal to deprioritize a page. Freshness age is a real but weak decline
signal on its own (a 10-point lift over base rate) -- worth using as one
input among several, not a standalone refresh trigger. Longer content
shows a directional traffic advantage, but it's moderate, not a case for
padding word count without a content-quality reason.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.